# Titanic — Exploratory Data Analysis

## Цель

Систематически исследовать тренировочную и тестовую выборки, проверить качество данных, изучить факторы выживаемости, взаимодействия признаков и сформировать каталог гипотез для последующего Feature Engineering.

Ноутбук предназначен для исследования и формирования гипотез. Все статистики, рассчитанные по полной тренировочной выборке, используются только для EDA и не должны автоматически переноситься в обучающий Pipeline.

Для загрузки данных запускай Jupyter из корня репозитория. Ожидаемые файлы: `data/raw/train.csv` и `data/raw/test.csv`.

## 1. Импорты и настройки визуализации

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, ks_2samp, mannwhitneyu
from scipy.spatial.distance import jensenshannon
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 42
TARGET = "Survived"
ID_COLUMN = "PassengerId"

## 2. Загрузка данных

In [ ]:
DATA_DIR = Path("/home/ecnedaced/projects/kaggle-titanic/data/raw")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train = train_raw.copy(deep=True)
test = test_raw.copy(deep=True)

train.shape, test.shape

In [ ]:
display(train.head())
display(test.head())

In [ ]:
dataset_overview = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [len(train), len(test)],
        "columns": [train.shape[1], test.shape[1]],
        "memory_mb": [train.memory_usage(deep=True).sum() / 1024**2, test.memory_usage(deep=True).sum() / 1024**2],
    }
)
display(dataset_overview)

In [ ]:
schema_comparison = pd.DataFrame(
    {
        "train_dtype": train.drop(columns=[TARGET]).dtypes.astype(str),
        "test_dtype": test.dtypes.astype(str),
    }
)
display(schema_comparison)

train_only_columns = sorted(set(train.columns) - set(test.columns))
test_only_columns = sorted(set(test.columns) - set(train.columns))
train_only_columns, test_only_columns

## 3. Первичный аудит качества данных

In [ ]:
quality_summary = pd.DataFrame(
    {
        "train": [
            len(train),
            train.shape[1],
            train.duplicated().sum(),
            train[ID_COLUMN].duplicated().sum(),
            train[ID_COLUMN].nunique(),
            train[TARGET].isna().sum(),
            train[TARGET].nunique(),
        ],
        "test": [
            len(test),
            test.shape[1],
            test.duplicated().sum(),
            test[ID_COLUMN].duplicated().sum(),
            test[ID_COLUMN].nunique(),
            np.nan,
            np.nan,
        ],
    },
    index=["rows", "columns", "duplicate_rows", "duplicate_ids", "unique_ids", "missing_target", "target_classes"],
)
display(quality_summary)

In [ ]:
dtype_summary = pd.DataFrame(
    {
        "dtype": train.dtypes.astype(str),
        "unique_values": train.nunique(dropna=True),
        "missing_count": train.isna().sum(),
        "missing_pct": train.isna().mean().mul(100),
    }
).sort_values(["missing_pct", "unique_values"], ascending=[False, False])
display(dtype_summary)

In [ ]:
missing_summary = pd.DataFrame(
    {
        "train_missing": train.reindex(columns=test.columns).isna().sum(),
        "train_missing_pct": train.reindex(columns=test.columns).isna().mean().mul(100),
        "test_missing": test.isna().sum(),
        "test_missing_pct": test.isna().mean().mul(100),
    }
).sort_values("train_missing_pct", ascending=False)
display(missing_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(color="steelblue", data=missing_summary.reset_index(names="feature"), x="train_missing_pct", y="feature", ax=axes[0])
axes[0].set_title("Пропуски в train")
axes[0].set_xlabel("Доля пропусков, %")
axes[0].set_ylabel("")
sns.barplot(color="steelblue", data=missing_summary.reset_index(names="feature"), x="test_missing_pct", y="feature", ax=axes[1])
axes[1].set_title("Пропуски в test")
axes[1].set_xlabel("Доля пропусков, %")
axes[1].set_ylabel("")
plt.tight_layout()

### 3.1. Распределение целевой переменной

In [ ]:
target_distribution = train[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count")
target_distribution["share"] = target_distribution["count"] / len(train)
display(target_distribution)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(color="steelblue", data=train, x=TARGET, ax=ax)
ax.set_title("Распределение Survived")
ax.set_xlabel("Survived")
ax.set_ylabel("Количество пассажиров")
plt.tight_layout()

### 3.2. Анализ пропусков в разрезе целевой переменной

In [ ]:
missing_by_target = (
    train.assign(
        AgeMissing=train["Age"].isna(),
        CabinMissing=train["Cabin"].isna(),
        EmbarkedMissing=train["Embarked"].isna(),
    )
    .groupby(TARGET)[["AgeMissing", "CabinMissing", "EmbarkedMissing"]]
    .mean()
    .mul(100)
)
display(missing_by_target)

missing_by_sex_class = (
    train.assign(AgeMissing=train["Age"].isna(), CabinMissing=train["Cabin"].isna())
    .groupby(["Sex", "Pclass"])[["AgeMissing", "CabinMissing"]]
    .mean()
    .mul(100)
)
display(missing_by_sex_class)

## 4. Одномерные распределения исходных признаков

### 4.1. Категориальные признаки

In [ ]:
categorical_features = ["Pclass", "Sex", "Embarked"]

for feature in categorical_features:
    counts = train[feature].value_counts(dropna=False).rename_axis(feature).to_frame("count")
    counts["share"] = counts["count"] / len(train)
    display(counts)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.countplot(color="steelblue", data=train, x=feature, order=train[feature].value_counts().index, ax=ax)
    ax.set_title(f"Распределение {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("Количество пассажиров")
    plt.tight_layout()

### 4.2. Числовые признаки

In [ ]:
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
display(train[numeric_features].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for feature, ax in zip(numeric_features, axes.flat):
    sns.histplot(color="steelblue", data=train, x=feature, kde=True, bins=30, ax=ax)
    ax.set_title(f"Распределение {feature}")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for feature, ax in zip(numeric_features, axes.flat):
    sns.boxplot(color="steelblue", data=train, x=feature, ax=ax)
    ax.set_title(f"Boxplot: {feature}")
plt.tight_layout()

## 5. Базовые связи признаков с выживаемостью

In [ ]:
def wilson_summary(frame, group_columns):
    result = (
        frame.groupby(group_columns, dropna=False, observed=True)
        .agg(n=(TARGET, "size"), survivors=(TARGET, "sum"))
        .reset_index()
    )
    z_value = 1.96
    proportion = result["survivors"] / result["n"]
    denominator = 1 + z_value**2 / result["n"]
    center = (proportion + z_value**2 / (2 * result["n"])) / denominator
    half_width = (
        z_value
        * np.sqrt(proportion * (1 - proportion) / result["n"] + z_value**2 / (4 * result["n"] ** 2))
        / denominator
    )
    result["survival_rate"] = proportion
    result["ci_lower"] = center - half_width
    result["ci_upper"] = center + half_width
    return result.sort_values("survival_rate", ascending=False)

survival_by_sex = wilson_summary(train, ["Sex"])
survival_by_class = wilson_summary(train, ["Pclass"])
survival_by_embarked = wilson_summary(train, ["Embarked"])
display(survival_by_sex)
display(survival_by_class)
display(survival_by_embarked)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for result, feature, ax in zip(
    [survival_by_sex, survival_by_class, survival_by_embarked],
    ["Sex", "Pclass", "Embarked"],
    axes,
):
    sns.barplot(color="steelblue", data=result, x=feature, y="survival_rate", ax=ax)
    ax.errorbar(
        x=np.arange(len(result)),
        y=result["survival_rate"],
        yerr=[result["survival_rate"] - result["ci_lower"], result["ci_upper"] - result["survival_rate"]],
        fmt="none",
        capsize=4,
        color="black",
    )
    ax.set_ylim(0, 1)
    ax.set_title(f"Survival Rate by {feature}")
    ax.set_ylabel("Доля выживших")
plt.tight_layout()

## 6. Возраст: распределение, нелинейность и взаимодействия

In [ ]:
age_summary = train.groupby(TARGET)["Age"].agg(["count", "mean", "median", "std", "min", "max"])
display(age_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(color="steelblue", data=train, x="Age", hue=TARGET, bins=30, stat="density", common_norm=False, element="step", ax=axes[0])
sns.boxplot(color="steelblue", data=train, x=TARGET, y="Age", ax=axes[1])
axes[0].set_title("Распределение возраста по Survived")
axes[1].set_title("Возраст по целевому классу")
plt.tight_layout()

In [ ]:
age_bins = [0, 5, 12, 18, 30, 45, 60, 80, 100]
age_labels = ["0-5", "6-12", "13-18", "19-30", "31-45", "46-60", "61-80", "81+"]
train["AgeGroup_EDA"] = pd.cut(train["Age"], bins=age_bins, labels=age_labels, include_lowest=True)

survival_by_age_group = wilson_summary(train, ["AgeGroup_EDA"])
display(survival_by_age_group)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(color="steelblue", data=survival_by_age_group, x="AgeGroup_EDA", y="survival_rate", order=age_labels, ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate by Age Group")
ax.set_xlabel("Возрастная группа")
ax.set_ylabel("Доля выживших")
plt.tight_layout()

In [ ]:
age_sex_table = (
    train.groupby(["AgeGroup_EDA", "Sex"], observed=True)[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
display(age_sex_table)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=age_sex_table, x="AgeGroup_EDA", y="survival_rate", hue="Sex", marker="o", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate: Age Group × Sex")
ax.set_xlabel("Возрастная группа")
ax.set_ylabel("Доля выживших")
plt.tight_layout()

In [ ]:
age_class_table = (
    train.groupby(["AgeGroup_EDA", "Pclass"], observed=True)[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
display(age_class_table)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=age_class_table, x="AgeGroup_EDA", y="survival_rate", hue="Pclass", marker="o", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate: Age Group × Pclass")
ax.set_xlabel("Возрастная группа")
ax.set_ylabel("Доля выживших")
plt.tight_layout()

In [ ]:
children_summary = wilson_summary(train.assign(IsChild_EDA=(train["Age"] < 16).astype("int8")), ["IsChild_EDA"])
display(children_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(color="steelblue", data=children_summary, x="IsChild_EDA", y="survival_rate", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate: Child vs Adult")
ax.set_xlabel("IsChild_EDA")
ax.set_ylabel("Доля выживших")
plt.tight_layout()

## 7. Fare: асимметрия, выбросы и относительная стоимость

In [ ]:
fare_summary = train["Fare"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame().T
display(fare_summary)

train["FareLog_EDA"] = np.log1p(train["Fare"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(color="steelblue", data=train, x="Fare", bins=40, kde=True, ax=axes[0])
sns.histplot(color="steelblue", data=train, x="FareLog_EDA", bins=40, kde=True, ax=axes[1])
axes[0].set_title("Fare: исходное распределение")
axes[1].set_title("Fare: log1p преобразование")
plt.tight_layout()

In [ ]:
fare_q1 = train["Fare"].quantile(0.25)
fare_q3 = train["Fare"].quantile(0.75)
fare_iqr = fare_q3 - fare_q1
fare_lower_bound = fare_q1 - 1.5 * fare_iqr
fare_upper_bound = fare_q3 + 1.5 * fare_iqr

fare_outlier_summary = pd.Series(
    {
        "q1": fare_q1,
        "q3": fare_q3,
        "iqr": fare_iqr,
        "lower_bound": fare_lower_bound,
        "upper_bound": fare_upper_bound,
        "outlier_count": ((train["Fare"] < fare_lower_bound) | (train["Fare"] > fare_upper_bound)).sum(),
        "outlier_share": ((train["Fare"] < fare_lower_bound) | (train["Fare"] > fare_upper_bound)).mean(),
    }
).to_frame("value")
display(fare_outlier_summary)

In [ ]:
fare_by_class = (
    train.groupby("Pclass")["Fare"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_index()
)
display(fare_by_class)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(color="steelblue", data=train, x="Pclass", y="Fare", ax=axes[0])
sns.boxplot(color="steelblue", data=train, x="Pclass", y="FareLog_EDA", ax=axes[1])
axes[0].set_title("Fare by Pclass")
axes[1].set_title("log1p(Fare) by Pclass")
plt.tight_layout()

In [ ]:
train["FareQuantile_EDA"] = pd.qcut(train["Fare"], q=5, duplicates="drop")
survival_by_fare_quantile = wilson_summary(train, ["FareQuantile_EDA"])
display(survival_by_fare_quantile)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(color="steelblue", data=survival_by_fare_quantile, x="FareQuantile_EDA", y="survival_rate", ax=ax)
ax.tick_params(axis="x", rotation=25)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate by Fare Quantile")
plt.tight_layout()

## 8. Name: титулы, фамилии и длина имени

In [ ]:
train["Title_EDA"] = train["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
train["Surname_EDA"] = train["Name"].str.split(",", n=1).str[0].str.strip()
train["NameLength_EDA"] = train["Name"].str.len()

train["TitleNormalized_EDA"] = train["Title_EDA"].replace(
    {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
        "Dona": "Mrs",
        "Don": "Mr",
        "Lady": "Royalty",
        "Countess": "Royalty",
        "Sir": "Royalty",
        "Jonkheer": "Royalty",
        "Capt": "Military",
        "Col": "Military",
        "Major": "Military",
    }
)

title_counts = train["Title_EDA"].value_counts(dropna=False).rename_axis("Title").to_frame("count")
title_counts["share"] = title_counts["count"] / len(train)
display(title_counts)

In [ ]:
survival_by_title = wilson_summary(train, ["Title_EDA"])
display(survival_by_title)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(color="steelblue", data=survival_by_title, x="Title_EDA", y="survival_rate", ax=ax)
ax.tick_params(axis="x", rotation=45)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate by Original Title")
plt.tight_layout()

In [ ]:
survival_by_normalized_title = wilson_summary(train, ["TitleNormalized_EDA"])
display(survival_by_normalized_title)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(color="steelblue", data=survival_by_normalized_title, x="TitleNormalized_EDA", y="survival_rate", ax=ax)
ax.tick_params(axis="x", rotation=35)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate by Normalized Title")
plt.tight_layout()

In [ ]:
name_length_summary = train.groupby(TARGET)["NameLength_EDA"].agg(["count", "mean", "median", "std"])
display(name_length_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(color="steelblue", data=train, x="NameLength_EDA", hue=TARGET, bins=30, stat="density", common_norm=False, ax=axes[0])
sns.boxplot(color="steelblue", data=train, x=TARGET, y="NameLength_EDA", ax=axes[1])
axes[0].set_title("Длина имени по Survived")
axes[1].set_title("Длина имени по целевому классу")
plt.tight_layout()

In [ ]:
train["SurnameGroupSize_EDA"] = train.groupby("Surname_EDA")["Surname_EDA"].transform("size")
surname_group_summary = train["SurnameGroupSize_EDA"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("value")
display(surname_group_summary)
display(
    train.loc[train["SurnameGroupSize_EDA"] > 1, ["Surname_EDA", "Name", "Ticket", "Pclass", "Survived", "SurnameGroupSize_EDA"]]
    .sort_values(["SurnameGroupSize_EDA", "Surname_EDA"], ascending=[False, True])
    .head(30)
)

## 9. Ticket: общие билеты, префиксы и размер билетной группы

In [ ]:
train["TicketNormalized_EDA"] = (
    train["Ticket"].astype("string").str.upper().str.replace(r"[./]", "", regex=True).str.replace(r"\s+", "", regex=True)
)
train["TicketPrefix_EDA"] = train["TicketNormalized_EDA"].str.extract(r"^([A-Z]+)", expand=False).fillna("NUMERIC_OR_EMPTY")
train["TicketIsNumeric_EDA"] = train["TicketNormalized_EDA"].str.fullmatch(r"\d+").fillna(False).astype("int8")
train["TicketGroupSize_EDA"] = train.groupby("Ticket", dropna=False)["Ticket"].transform("size")
train["FarePerTicketMember_EDA"] = train["Fare"] / train["TicketGroupSize_EDA"]

ticket_prefix_counts = train["TicketPrefix_EDA"].value_counts().rename_axis("TicketPrefix").to_frame("count")
ticket_prefix_counts["share"] = ticket_prefix_counts["count"] / len(train)
display(ticket_prefix_counts.head(30))

In [ ]:
survival_by_ticket_prefix = wilson_summary(train, ["TicketPrefix_EDA"])
display(survival_by_ticket_prefix.sort_values("n", ascending=False))

fig, ax = plt.subplots(figsize=(14, 6))
ticket_prefix_plot = survival_by_ticket_prefix.sort_values("n", ascending=False).head(20)
sns.barplot(color="steelblue", data=ticket_prefix_plot, x="TicketPrefix_EDA", y="survival_rate", ax=ax)
ax.tick_params(axis="x", rotation=50)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate by Frequent Ticket Prefixes")
plt.tight_layout()

In [ ]:
ticket_group_summary = train.groupby("TicketGroupSize_EDA")[TARGET].agg(["count", "mean"]).rename(columns={"count": "n", "mean": "survival_rate"})
display(ticket_group_summary)

fare_ticket_table = train[["Ticket", "TicketGroupSize_EDA", "Fare", "FarePerTicketMember_EDA", "Pclass", TARGET]].sort_values(
    ["TicketGroupSize_EDA", "Ticket"], ascending=[False, True]
)
display(fare_ticket_table.head(30))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(color="steelblue", data=train, x="TicketGroupSize_EDA", y="Fare", ax=axes[0])
sns.boxplot(color="steelblue", data=train, x="TicketGroupSize_EDA", y="FarePerTicketMember_EDA", ax=axes[1])
axes[0].set_title("Fare by Ticket Group Size")
axes[1].set_title("Fare per Ticket Member")
plt.tight_layout()

## 10. Cabin: палуба, наличие и количество обозначений

In [ ]:
train["HasCabin_EDA"] = train["Cabin"].notna().astype("int8")
train["CabinDeck_EDA"] = train["Cabin"].str[0].str.upper().fillna("Unknown")
train["CabinCount_EDA"] = train["Cabin"].str.split().str.len().fillna(0).astype("int8")

cabin_summary = pd.DataFrame(
    {
        "count": train["CabinDeck_EDA"].value_counts(dropna=False),
        "survival_rate": train.groupby("CabinDeck_EDA")[TARGET].mean(),
    }
).sort_values("count", ascending=False)
display(cabin_summary)

In [ ]:
survival_by_deck = wilson_summary(train, ["CabinDeck_EDA"])
display(survival_by_deck)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(color="steelblue", data=train, x="CabinDeck_EDA", order=train["CabinDeck_EDA"].value_counts().index, ax=axes[0])
sns.barplot(color="steelblue", data=survival_by_deck, x="CabinDeck_EDA", y="survival_rate", ax=axes[1])
axes[0].set_title("Количество пассажиров по палубам")
axes[1].set_title("Survival Rate по палубам")
axes[1].set_ylim(0, 1)
plt.tight_layout()

In [ ]:
deck_class_table = (
    train.groupby(["CabinDeck_EDA", "Pclass"])[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
display(deck_class_table)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(color="steelblue", data=deck_class_table, x="CabinDeck_EDA", y="survival_rate", hue="Pclass", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Survival Rate: Deck × Pclass")
plt.tight_layout()

In [ ]:
cabin_presence = wilson_summary(train, ["HasCabin_EDA"])
cabin_count_survival = (
    train.groupby("CabinCount_EDA")[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
)
display(cabin_presence)
display(cabin_count_survival)

## 11. Family: размер семьи и взаимодействия

In [ ]:
train["FamilySize_EDA"] = train["SibSp"] + train["Parch"] + 1
train["IsAlone_EDA"] = (train["FamilySize_EDA"] == 1).astype("int8")
train["FamilyGroup_EDA"] = pd.cut(
    train["FamilySize_EDA"],
    bins=[0, 1, 4, np.inf],
    labels=["Alone", "Small", "Large"],
    include_lowest=True,
)

family_size_counts = train["FamilySize_EDA"].value_counts().sort_index().to_frame("count")
family_size_counts["share"] = family_size_counts["count"] / len(train)
display(family_size_counts)
display(wilson_summary(train, ["FamilyGroup_EDA"]))
display(wilson_summary(train, ["IsAlone_EDA"]))

In [ ]:
sibsp_survival = wilson_summary(train, ["SibSp"])
parch_survival = wilson_summary(train, ["Parch"])
family_size_survival = wilson_summary(train, ["FamilySize_EDA"])
display(sibsp_survival)
display(parch_survival)
display(family_size_survival)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for result, feature, ax in zip(
    [sibsp_survival, parch_survival, family_size_survival],
    ["SibSp", "Parch", "FamilySize_EDA"],
    axes,
):
    sns.barplot(color="steelblue", data=result, x=feature, y="survival_rate", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_title(f"Survival Rate by {feature}")
plt.tight_layout()

In [ ]:
family_sex_table = (
    train.groupby(["FamilyGroup_EDA", "Sex"])[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
family_age_table = (
    train.groupby(["FamilyGroup_EDA", "AgeGroup_EDA"], observed=True)[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
display(family_sex_table)
display(family_age_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(color="steelblue", data=family_sex_table, x="FamilyGroup_EDA", y="survival_rate", hue="Sex", ax=axes[0])
sns.barplot(color="steelblue", data=family_age_table, x="AgeGroup_EDA", y="survival_rate", hue="FamilyGroup_EDA", ax=axes[1])
axes[0].set_ylim(0, 1)
axes[1].set_ylim(0, 1)
axes[0].set_title("Survival Rate: Family Group × Sex")
axes[1].set_title("Survival Rate: Family Group × Age")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()

## 12. Embarked и совместные эффекты

In [ ]:
embarked_class_table = (
    train.groupby(["Embarked", "Pclass"], dropna=False)[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
embarked_sex_table = (
    train.groupby(["Embarked", "Sex"], dropna=False)[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
display(embarked_class_table)
display(embarked_sex_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(color="steelblue", data=embarked_class_table, x="Embarked", y="survival_rate", hue="Pclass", ax=axes[0])
sns.barplot(color="steelblue", data=embarked_sex_table, x="Embarked", y="survival_rate", hue="Sex", ax=axes[1])
axes[0].set_ylim(0, 1)
axes[1].set_ylim(0, 1)
axes[0].set_title("Survival Rate: Embarked × Pclass")
axes[1].set_title("Survival Rate: Embarked × Sex")
plt.tight_layout()

## 13. PassengerId и порядок наблюдений

In [ ]:
train["PassengerIdBin_EDA"] = pd.qcut(train[ID_COLUMN], q=10, duplicates="drop")
id_bin_survival = (
    train.groupby("PassengerIdBin_EDA", observed=True)[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "survival_rate"})
    .reset_index()
)
display(id_bin_survival)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(np.arange(len(id_bin_survival)), id_bin_survival["survival_rate"], marker="o")
ax.set_xticks(np.arange(len(id_bin_survival)))
ax.set_xticklabels(id_bin_survival["PassengerIdBin_EDA"].astype(str), rotation=30, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Survival Rate across PassengerId quantiles")
plt.tight_layout()

## 14. Корреляции и ассоциации признаков

In [ ]:
correlation_features = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize_EDA", "NameLength_EDA", "TicketGroupSize_EDA"]
pearson_correlation = train[correlation_features].corr(method="pearson", numeric_only=True)
spearman_correlation = train[correlation_features].corr(method="spearman", numeric_only=True)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
sns.heatmap(pearson_correlation, annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[0])
sns.heatmap(spearman_correlation, annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1])
axes[0].set_title("Pearson correlation")
axes[1].set_title("Spearman correlation")
plt.tight_layout()

In [ ]:
def cramers_v(frame, first_column, second_column):
    contingency_table = pd.crosstab(frame[first_column], frame[second_column])
    chi_square = chi2_contingency(contingency_table, correction=False)[0]
    sample_size = contingency_table.to_numpy().sum()
    row_count, column_count = contingency_table.shape
    phi_squared = chi_square / sample_size
    correction = ((column_count - 1) * (row_count - 1)) / (sample_size - 1)
    phi_squared_corrected = max(0, phi_squared - correction)
    row_corrected = row_count - ((row_count - 1) ** 2) / (sample_size - 1)
    column_corrected = column_count - ((column_count - 1) ** 2) / (sample_size - 1)
    return np.sqrt(phi_squared_corrected / min(column_corrected - 1, row_corrected - 1))

categorical_associations = pd.DataFrame(
    [
        {"feature": feature, "cramers_v_with_survived": cramers_v(train, feature, TARGET)}
        for feature in ["Sex", "Pclass", "Embarked", "TitleNormalized_EDA", "TicketPrefix_EDA", "CabinDeck_EDA", "FamilyGroup_EDA"]
    ]
).sort_values("cramers_v_with_survived", ascending=False)
display(categorical_associations)

In [ ]:
numeric_mi_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize_EDA", "NameLength_EDA", "TicketGroupSize_EDA", "FarePerTicketMember_EDA"]
mi_numeric_input = train[numeric_mi_features].copy()
mi_numeric_input = mi_numeric_input.fillna(mi_numeric_input.median())
mi_numeric_scores = mutual_info_classif(
    mi_numeric_input,
    train[TARGET],
    discrete_features=False,
    random_state=RANDOM_STATE,
)

numeric_mutual_information = pd.DataFrame(
    {"feature": numeric_mi_features, "mutual_information": mi_numeric_scores}
).sort_values("mutual_information", ascending=False)
display(numeric_mutual_information)

In [ ]:
categorical_mi_features = ["Sex", "Pclass", "Embarked", "TitleNormalized_EDA", "TicketPrefix_EDA", "CabinDeck_EDA", "FamilyGroup_EDA"]

categorical_mi_results = []
for feature in categorical_mi_features:
    encoded_feature = train[feature].astype("string").fillna("Missing").astype("category").cat.codes.to_frame()
    score = mutual_info_classif(
        encoded_feature,
        train[TARGET],
        discrete_features=True,
        random_state=RANDOM_STATE,
    )[0]
    categorical_mi_results.append({"feature": feature, "mutual_information": score})

categorical_mutual_information = pd.DataFrame(categorical_mi_results).sort_values("mutual_information", ascending=False)
display(categorical_mutual_information)

## 15. Статистические тесты для первичных гипотез

In [ ]:
age_survivors = train.loc[train[TARGET] == 1, "Age"].dropna()
age_non_survivors = train.loc[train[TARGET] == 0, "Age"].dropna()
fare_survivors = train.loc[train[TARGET] == 1, "Fare"].dropna()
fare_non_survivors = train.loc[train[TARGET] == 0, "Fare"].dropna()

mann_whitney_results = pd.DataFrame(
    [
        {
            "feature": "Age",
            "statistic": mannwhitneyu(age_survivors, age_non_survivors, alternative="two-sided").statistic,
            "p_value": mannwhitneyu(age_survivors, age_non_survivors, alternative="two-sided").pvalue,
        },
        {
            "feature": "Fare",
            "statistic": mannwhitneyu(fare_survivors, fare_non_survivors, alternative="two-sided").statistic,
            "p_value": mannwhitneyu(fare_survivors, fare_non_survivors, alternative="two-sided").pvalue,
        },
    ]
)
display(mann_whitney_results)

In [ ]:
chi_square_features = ["Sex", "Pclass", "Embarked", "TitleNormalized_EDA", "FamilyGroup_EDA", "CabinDeck_EDA"]
chi_square_results = []
for feature in chi_square_features:
    contingency_table = pd.crosstab(train[feature], train[TARGET])
    statistic, p_value, degrees_of_freedom, expected = chi2_contingency(contingency_table)
    chi_square_results.append(
        {
            "feature": feature,
            "chi2": statistic,
            "degrees_of_freedom": degrees_of_freedom,
            "p_value": p_value,
            "cramers_v": cramers_v(train, feature, TARGET),
        }
    )

chi_square_results = pd.DataFrame(chi_square_results).sort_values("p_value")
display(chi_square_results)

## 16. Сравнение train и test: сдвиг распределений

Сравнение train/test помогает заметить covariate shift и различия в заполнении данных. Оно не использует test target: в соревновании целевой столбец для test недоступен.

In [ ]:
drift_numeric_features = ["Age", "SibSp", "Parch", "Fare"]
numeric_drift_results = []
for feature in drift_numeric_features:
    train_values = train[feature].dropna()
    test_values = test[feature].dropna()
    pooled_std = np.sqrt((train_values.var() + test_values.var()) / 2)
    numeric_drift_results.append(
        {
            "feature": feature,
            "train_mean": train_values.mean(),
            "test_mean": test_values.mean(),
            "train_median": train_values.median(),
            "test_median": test_values.median(),
            "standardized_mean_difference": (test_values.mean() - train_values.mean()) / pooled_std,
            "ks_statistic": ks_2samp(train_values, test_values).statistic,
            "ks_p_value": ks_2samp(train_values, test_values).pvalue,
        }
    )

numeric_drift = pd.DataFrame(numeric_drift_results).sort_values("ks_statistic", ascending=False)
display(numeric_drift)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for feature, ax in zip(drift_numeric_features, axes.flat):
    sns.kdeplot(data=train, color="steelblue", x=feature, label="train", fill=False, ax=ax)
    sns.kdeplot(data=test, color="darkorange", x=feature, label="test", fill=False, ax=ax)
    ax.set_title(f"Train/Test: {feature}")
    ax.legend()
plt.tight_layout()

In [ ]:
def categorical_drift(train_series, test_series):
    train_values = train_series.astype("string").fillna("Missing")
    test_values = test_series.astype("string").fillna("Missing")
    categories = pd.Index(train_values.unique()).union(pd.Index(test_values.unique()))
    train_distribution = train_values.value_counts(normalize=True).reindex(categories, fill_value=0).to_numpy()
    test_distribution = test_values.value_counts(normalize=True).reindex(categories, fill_value=0).to_numpy()
    return jensenshannon(train_distribution, test_distribution, base=2)

categorical_drift_features = ["Pclass", "Sex", "Embarked"]
categorical_drift = pd.DataFrame(
    [
        {
            "feature": feature,
            "train_unique": train[feature].nunique(dropna=False),
            "test_unique": test[feature].nunique(dropna=False),
            "jensen_shannon_distance": categorical_drift(train[feature], test[feature]),
        }
        for feature in categorical_drift_features
    ]
).sort_values("jensen_shannon_distance", ascending=False)
display(categorical_drift)

## 17. Инженерия признаков: исследовательский пул кандидатов

Ниже создаются только EDA-копии признаков. Исходные `train_raw` и `test_raw` остаются без изменений. Статистики по целевой переменной используются для анализа связей, но не создаются как признаки.

In [ ]:
eda_features = train_raw.copy()
eda_features["FamilySize"] = eda_features["SibSp"] + eda_features["Parch"] + 1
eda_features["IsAlone"] = (eda_features["FamilySize"] == 1).astype("int8")
eda_features["FareLog"] = np.log1p(eda_features["Fare"])
eda_features["FarePerPerson"] = eda_features["Fare"] / eda_features["FamilySize"]
eda_features["Title"] = eda_features["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
eda_features["NameLength"] = eda_features["Name"].str.len()
eda_features["Surname"] = eda_features["Name"].str.split(",", n=1).str[0].str.strip()
eda_features["TicketPrefix"] = (
    eda_features["Ticket"].astype("string").str.upper().str.replace(r"[./]", "", regex=True).str.replace(r"\s+", "", regex=True)
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("NUMERIC_OR_EMPTY")
)
eda_features["CabinDeck"] = eda_features["Cabin"].str[0].str.upper().fillna("Unknown")
eda_features["HasCabin"] = eda_features["Cabin"].notna().astype("int8")
eda_features["AgeMissing"] = eda_features["Age"].isna().astype("int8")
eda_features["CabinMissing"] = eda_features["Cabin"].isna().astype("int8")
eda_features["EmbarkedMissing"] = eda_features["Embarked"].isna().astype("int8")

candidate_feature_summary = eda_features[
    ["FamilySize", "IsAlone", "FareLog", "FarePerPerson", "Title", "NameLength", "Surname", "TicketPrefix", "CabinDeck", "HasCabin", "AgeMissing", "CabinMissing", "EmbarkedMissing"]
].describe(include="all").T
display(candidate_feature_summary)

In [ ]:
candidate_numeric_features = ["FamilySize", "IsAlone", "FareLog", "FarePerPerson", "NameLength", "HasCabin", "AgeMissing", "CabinMissing", "EmbarkedMissing"]
candidate_numeric_mi_input = eda_features[candidate_numeric_features].copy()
candidate_numeric_mi_input = candidate_numeric_mi_input.fillna(candidate_numeric_mi_input.median())
candidate_numeric_mi_scores = mutual_info_classif(
    candidate_numeric_mi_input,
    eda_features[TARGET],
    discrete_features=[False, True, False, False, False, True, True, True, True],
    random_state=RANDOM_STATE,
)

candidate_numeric_mi = pd.DataFrame(
    {"feature": candidate_numeric_features, "mutual_information": candidate_numeric_mi_scores}
).sort_values("mutual_information", ascending=False)
display(candidate_numeric_mi)

## 18. Каталог гипотез для Feature Engineering

In [ ]:
feature_hypotheses = pd.DataFrame(
    [
        ["Title", "Извлекается из Name", "Stateless", "Высокая", "Редкие титулы и нормализация проверяются на train; не кодировать частоты по всему датасету до CV"],
        ["FamilySize", "SibSp + Parch + 1", "Stateless", "Высокая", "Построчная арифметика"],
        ["IsAlone", "FamilySize == 1", "Stateless", "Высокая", "Построчная логика"],
        ["FareLog", "log1p(Fare)", "Stateless", "Средняя", "Построчное преобразование; границы/масштабирование не обучаются здесь"],
        ["FarePerPerson", "Fare / FamilySize", "Stateless", "Средняя", "Построчная арифметика; смысл Fare требует отдельной проверки"],
        ["TicketPrefix", "Префикс Ticket", "Stateless", "Средняя", "Нормализация строки на уровне записи"],
        ["TicketGroupSize", "Частота Ticket", "Fold-dependent", "Высокая", "Групповые частоты должны вычисляться без доступа к validation target и с корректной обработкой unseen групп"],
        ["FarePerTicketMember", "Fare / TicketGroupSize", "Fold-dependent", "Средняя", "Зависит от групповой частоты Ticket"],
        ["CabinDeck", "Первая буква Cabin", "Stateless", "Средняя", "Построчное извлечение строки"],
        ["HasCabin", "Cabin is not null", "Stateless", "Высокая", "Индикатор пропуска на уровне записи"],
        ["CabinCount", "Количество обозначений Cabin", "Stateless", "Низкая", "Построчный разбор строки"],
        ["NameLength", "Длина Name", "Stateless", "Низкая", "Построчная длина строки"],
        ["SurnameGroupSize", "Частота фамилии", "Fold-dependent", "Средняя", "Групповые частоты должны быть безопасны внутри CV"],
        ["AgeMissing", "Age is null", "Stateless", "Средняя", "Индикатор пропуска на уровне записи"],
        ["AgeImputation", "Импутация Age", "Stateful", "Высокая", "Импутер fit только на train-fold"],
        ["FareImputation", "Импутация Fare", "Stateful", "Средняя", "Импутер fit только на train-fold"],
        ["RareCategoryGrouping", "Группировка редких категорий", "Stateful", "Средняя", "Порог и список категорий fit только на train-fold"],
        ["Scaling", "StandardScaler / RobustScaler", "Stateful", "По модели", "Scaler fit только на train-fold"],
        ["TargetEncoding", "Кодирование средним Survived", "Target-dependent", "Экспериментальная", "Только out-of-fold схема с внутренним сглаживанием; глобальный target encoding запрещён"],
        ["Age × Sex", "Взаимодействие возраста и пола", "Stateless", "Средняя", "Взаимодействие строится построчно; возрастная импутация остаётся fold-safe"],
        ["Pclass × Fare", "Взаимодействие класса и тарифа", "Stateless", "Средняя", "Проверить нелинейность и влияние выбросов"],
        ["FamilySize × Sex", "Размер семьи и пол", "Stateless", "Средняя", "Построчная комбинация признаков"],
    ],
    columns=["feature", "definition", "feature_type", "priority", "leakage_and_validation_notes"],
)
display(feature_hypotheses)

## 19. Финальные выводы и решения для следующего этапа

После выполнения ноутбука заполни аналитические выводы по следующим пунктам:

1. Какие признаки имеют устойчивую связь с выживаемостью и достаточный размер групп.
2. Какие эффекты являются нелинейными и требуют биннинга или преобразований.
3. Какие взаимодействия дают содержательно заметные различия.
4. Какие признаки имеют существенные пропуски и связаны ли пропуски с другими переменными.
5. Насколько отличаются распределения train и test.
6. Какие кандидаты стоит включить в первую версию `FeatureGenerator`.
7. Какие преобразования обязательно должны обучаться внутри каждого CV-fold.

### Ограничения интерпретации

- EDA на полной тренировочной выборке не является оценкой обобщающей способности модели.
- p-value и Mutual Information здесь используются для генерации гипотез, а не как окончательный алгоритм отбора признаков.
- Множественные проверки гипотез повышают вероятность случайных находок.
- Небольшие группы и редкие категории требуют осторожной интерпретации доверительных интервалов.
- Любые статистики, зависящие от распределения выборки, должны вычисляться внутри соответствующего train-fold в ML Pipeline.
- `PassengerId` исследуется только как потенциальный артефакт порядка данных и не рассматривается как стандартный предиктор без отдельного обоснования.